In [ ]:
import os
import glob
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

In [ ]:
# =====================================================================
# 0. CONFIGURAZIONE E VERIFICA GPU
# =====================================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Consente a TF di allocare memoria sulla GPU man mano che serve, 
        # invece di allocare il 100% della VRAM istantaneamente (evita crash su cluster)
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"=== GPU Rilevata con successo! Disponibili: {len(gpus)} GPU ===")
        print(f"Dettagli: {gpus}")
    except RuntimeError as e:
        print(f"Errore nella configurazione della GPU: {e}")
else:
    print("=== [ATTENZIONE] Nessuna GPU rilevata. Il codice girerà in CPU (Molto lento) ===")

In [ ]:
# =====================================================================
# 1. HELPER FUNCTIONS
# =====================================================================

def load_all_images(folder, img_size=(128,128)):
    """Load grayscale images from folder, normalize globally to [0,1]."""
    exts = ['*.png', '*.PNG', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(folder, e)))
    paths = sorted(paths)
    imgs = []
    for p in paths:
        try:
            im = Image.open(p).convert("L")  # grayscale
            im = im.resize(img_size, Image.BILINEAR)
            arr = np.array(im, dtype=np.float32) / 255.0  # scale to [0,1]
            imgs.append(arr[..., None])  # add channel dim
        except Exception as e:
            print(f"[load_all_images] skipping {p}: {e}")
            
    if len(imgs) == 0:
        raise RuntimeError(f"No images loaded from {folder}")
    return np.stack(imgs, axis=0) 

def augment_tf(img):
    """Simple augmentation: random flips + 90° rotations."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    return img

In [ ]:
# =====================================================================
# 2. DATA ASSEMBLY & PEDESTAL MIXING PIPELINE
# =====================================================================

print("=== Loading Datasets ===")

# A. Load Real Data
print("Loading real background (STD)...", end="", flush=True)
std_images_real = load_all_images("Dataset_Preparation/STD_images_train")
print(f" Done! ({len(std_images_real)} images)")

print("Loading real mixture (AmBe)...", end="", flush=True)
ambe_images_real = load_all_images("Dataset_Preparation/AmBe_images_train")
print(f" Done! ({len(ambe_images_real)} images)")

X_real = np.concatenate([std_images_real, ambe_images_real], axis=0)

# B. Load Simulated Data 
energies = [1, 3, 5, 10, 15, 20, 25, 35, 45, 50]
sim_base = "MC_Reconstructed_Dataset/output_images"

ER_simulated = []
NR_simulated = []

for energy in energies:
    er_folder = os.path.join(sim_base, f"{energy}_keV", f"image_CYGNO_60_40_ER_{energy}_keV")
    if os.path.exists(er_folder):
        print(f"Loading Sim ER ({energy} keV)...", end="", flush=True)
        imgs = load_all_images(er_folder)
        print(f" Done! ({len(imgs)} images)")
        ER_simulated.append(imgs)
        
    nr_folder = os.path.join(sim_base, f"{energy}_keV", f"image_CYGNO_60_40_He_{energy}_keVee")
    if os.path.exists(nr_folder):
        print(f"Loading Sim NR ({energy} keV)...", end="", flush=True)
        imgs = load_all_images(nr_folder)
        print(f" Done! ({len(imgs)} images)")
        NR_simulated.append(imgs)

# Salviamo questi array puliti per riciclarli nella Fase 2
X_sim_er = np.concatenate(ER_simulated, axis=0) if len(ER_simulated) > 0 else np.array([])
X_sim_nr = np.concatenate(NR_simulated, axis=0) if len(NR_simulated) > 0 else np.array([])

X_simulated = np.concatenate([X_sim_er, X_sim_nr], axis=0)

# C. Creazione Dataset Finale per Fase 1
X_all = np.concatenate([X_real, X_simulated], axis=0)
flags_real = np.zeros(len(X_real), dtype=np.float32)
flags_sim = np.ones(len(X_simulated), dtype=np.float32)
flags_all = np.concatenate([flags_real, flags_sim], axis=0)

print(f"\nFinal Dataset Size for Pre-training: {X_all.shape}")

# D. Pipeline tf.data con Background Invariance
BATCH_SIZE = 128 
STD_BANK = tf.constant(std_images_real, dtype=tf.float32)
NUM_STD_IMAGES = len(std_images_real)

def get_random_background():
    idx = tf.random.uniform(shape=[], minval=0, maxval=NUM_STD_IMAGES, dtype=tf.int32)
    return STD_BANK[idx]

def prepare_contrastive_pair(img, is_simulated_flag):
    view_i = augment_tf(img)
    view_j = augment_tf(img)
    
    def apply_mixing(view):
        bg = get_random_background()
        return tf.clip_by_value(view + bg, 0.0, 1.0)
    
    view_i = tf.cond(is_simulated_flag > 0, lambda: apply_mixing(view_i), lambda: view_i)
    view_j = tf.cond(is_simulated_flag > 0, lambda: apply_mixing(view_j), lambda: view_j)
    return view_i, view_j

dataset = tf.data.Dataset.from_tensor_slices((X_all, flags_all))
dataset = dataset.shuffle(buffer_size=1024)
dataset = dataset.map(prepare_contrastive_pair, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# =====================================================================
# 3. SIMCLR ARCHITECTURE & LOSS
# =====================================================================

def build_contrastive_network(input_shape=(128, 128, 1), embedding_dim=64):
    inputs = keras.Input(shape=input_shape)
    
    # --- ENCODER BACKBONE ---
    x = layers.Conv2D(32, 3, strides=1, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(64, 3, strides=1, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(128, 3, strides=1, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    encoder_features = layers.GlobalAveragePooling2D()(x)
    
    # --- PROJECTION HEAD ---
    p = layers.Dense(128, activation="relu")(encoder_features)
    projections = layers.Dense(embedding_dim, activation=None)(p) 
    
    return keras.Model(inputs, projections, name="simclr_network")

class SimCLRModel(keras.Model):
    def __init__(self, backbone, temperature=0.1):
        super().__init__()
        self.backbone = backbone
        self.temperature = temperature
        self.loss_tracker = keras.metrics.Mean(name="contrastive_loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def compute_nt_xent_loss(self, projections_i, projections_j):
        z_i = tf.math.l2_normalize(projections_i, axis=1)
        z_j = tf.math.l2_normalize(projections_j, axis=1)
        z = tf.concat([z_i, z_j], axis=0)
        
        similarity_matrix = tf.matmul(z, z, transpose_b=True)
        sim_matrix_scaled = similarity_matrix / self.temperature
        
        batch_size = tf.shape(projections_i)[0]
        mask = tf.cast(tf.eye(batch_size), tf.bool)
        positives_mask_left = tf.concat([tf.zeros_like(mask), mask], axis=1)
        positives_mask_right = tf.concat([mask, tf.zeros_like(mask)], axis=1)
        positives_mask = tf.concat([positives_mask_left, positives_mask_right], axis=0)
        
        labels_mask = tf.cast(tf.eye(2 * batch_size), tf.bool)
        exclusion_mask = tf.math.logical_or(labels_mask, positives_mask)
        negatives_mask = tf.math.logical_not(exclusion_mask)
        
        positives = tf.boolean_mask(sim_matrix_scaled, positives_mask)
        positives = tf.reshape(positives, (2 * batch_size, 1))
        negatives = tf.boolean_mask(sim_matrix_scaled, negatives_mask)
        negatives = tf.reshape(negatives, (2 * batch_size, (2 * batch_size) - 2)) 
        
        logits = tf.concat([positives, negatives], axis=1)
        labels = tf.zeros(2 * batch_size, dtype=tf.int32)
        
        return tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=logits)
        )

    @tf.function 
    def train_step(self, data):
        view_i, view_j = data
        with tf.GradientTape() as tape:
            projections_i = self.backbone(view_i, training=True)
            projections_j = self.backbone(view_j, training=True)
            loss = self.compute_nt_xent_loss(projections_i, projections_j)
            
        gradients = tape.gradient(loss, self.backbone.trainable_variables) 
        self.optimizer.apply_gradients(zip(gradients, self.backbone.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

In [ ]:
# =====================================================================
# 4. TRAINING EXECUTION (PHASE 1) - VERSIONE GPU
# =====================================================================

EPOCHS = 100
total_steps = len(dataset)
decay_steps = EPOCHS * total_steps

# Dinamic Learning Rate (Cosine Decay)
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, 
    decay_steps=decay_steps,
    alpha=0.0 
)

# --- FORZIAMO IL MODELLO SULLA GPU ---
with tf.device('/GPU:0'):
    print("Inizializzazione del modello SimCLR sulla GPU...")
    model = build_contrastive_network()
    optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)

    simclr_trainer = SimCLRModel(backbone=model, temperature=0.1)
    simclr_trainer.compile(optimizer=optimizer) 
# -------------------------------------

os.makedirs("saved_models", exist_ok=True) 

print(f"\n=== Starting Phase 1: Contrastive Pre-training ({EPOCHS} Epochs) ===")

best_loss = float('inf')
patience = 20 
patience_counter = 0
best_model_path = "saved_models/SimCLR_Best_Encoder.weights.h5"

for epoch in range(EPOCHS):
    start_time = time.time()
    simclr_trainer.loss_tracker.reset_states()
    current_lr = optimizer.learning_rate(epoch * total_steps).numpy()
    
    print(f"--- Starting Epoch {epoch+1}/{EPOCHS} (LR: {current_lr:.6f}) ---")
    
    for step, batch_data in enumerate(dataset):
        metrics = simclr_trainer.train_step(batch_data)
        if step % 10 == 0 or step == total_steps - 1:
            current_loss = metrics['loss'].numpy()
            print(f"  -> Batch {step}/{total_steps} | Partial Loss: {current_loss:.4f}", end="\r")
            
    epoch_time = time.time() - start_time
    final_epoch_loss = metrics["loss"].numpy()
    
    print(f"\n[Epoch {epoch+1} Complete] Loss: {final_epoch_loss:.4f} | Time: {epoch_time:.1f} sec")
    
    # Early Stopping Logic
    if final_epoch_loss < best_loss:
        print(f"  [+] Loss migliorata da {best_loss:.4f} a {final_epoch_loss:.4f}. Salvo il modello.")
        best_loss = final_epoch_loss
        patience_counter = 0 
        simclr_trainer.backbone.save_weights(best_model_path)
    else:
        patience_counter += 1
        print(f"  [-] Loss non migliorata. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"\n[!] EARLY STOPPING TRIGGERATO. Nessun miglioramento da {patience} epoche.")
        break 

print("\n=== Pre-training Complete! ===")

if os.path.exists(best_model_path):
    simclr_trainer.backbone.load_weights(best_model_path)
    print("Pesi del modello migliore ripristinati per il salvataggio finale.")

final_model_path = "saved_models/CYGNO_Encoder_SimCLR_Final.keras"
simclr_trainer.backbone.save(final_model_path)
print(f"Encoder model successfully saved to: {final_model_path}")

In [ ]:
# =====================================================================
# 5. LINEAR EVALUATION PROTOCOL (PHASE 2)
# =====================================================================

print("\n=== Assemblaggio Dataset per Fase 2 (da memoria RAM) ===")

# Assegnazione Label (0 = Background, 1 = Signal)
y_std_real = np.zeros(std_images_real.shape[0], dtype=np.int32)
y_sim_er   = np.zeros(X_sim_er.shape[0], dtype=np.int32)
y_sim_nr   = np.ones(X_sim_nr.shape[0], dtype=np.int32)

# Uniamo SOLO Background Puro e Segnale Puro (Escludiamo l'AmBe mischiato)
X_labeled = np.concatenate([std_images_real, X_sim_er, X_sim_nr], axis=0)
y_labeled = np.concatenate([y_std_real, y_sim_er, y_sim_nr], axis=0)

# Train/Val Split (Lo stratify garantisce che le proporzioni rimangano costanti)
X_train, X_val, y_train, y_val = train_test_split(
    X_labeled, 
    y_labeled, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_labeled 
)

print(f"Training set:   {X_train.shape[0]} images")
print(f"Validation set: {X_val.shape[0]} images")

# --- Creazione del Classificatore Lineare ---
# --- Creazione del Classificatore Lineare SU GPU ---
print("\nLoading pre-trained encoder...")

with tf.device('/GPU:0'):
    encoder = keras.models.load_model(final_model_path)
    encoder.trainable = False  # CRITICAL: Freeze the base network

    inputs = keras.Input(shape=(128, 128, 1))
    features = encoder(inputs, training=False) 
    outputs = layers.Dense(1, activation="sigmoid", name="linear_classifier")(features)

    linear_eval_model = keras.Model(inputs, outputs, name="SimCLR_Linear_Eval")

    linear_eval_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
# ----------------------------------------------------

# --- Calcolo dei Pesi per il Bilanciamento delle Classi ---
total_samples = len(y_train)
class_0_count = np.sum(y_train == 0)
class_1_count = np.sum(y_train == 1)

# Formula standard di bilanciamento
weight_for_0 = (1 / class_0_count) * (total_samples / 2.0)
weight_for_1 = (1 / class_1_count) * (total_samples / 2.0)

class_weights = {0: weight_for_0, 1: weight_for_1}

print("\n--- Bilanciamento Classi ---")
print(f"Background (0): {class_0_count} tracce -> Peso applicato: {weight_for_0:.2f}")
print(f"Segnali NR (1): {class_1_count} tracce -> Peso applicato: {weight_for_1:.2f}")

# --- Addestramento Classificatore ---
print("\nTraining Linear Classifier on frozen representations...")
history = linear_eval_model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50, 
    validation_data=(X_val, y_val),
    class_weight=class_weights # <--- Applicazione dei pesi calcolati
)

print("\n=== Linear Evaluation Complete ===")

# Salvataggio del classificatore finale
final_classifier_path = "saved_models/CYGNO_Linear_Classifier_Final.keras"
linear_eval_model.save(final_classifier_path)
print(f"Classification model successfully saved to: {final_classifier_path}")

In [ ]:
# =====================================================================
# 6. INFERENCE & EVALUATION OF DISTRIBUTIONS (PHASE 3)
# =====================================================================

print("=== Avvio Fase 3: Inferenza e Valutazione delle Distribuzioni ===")

# 1. Calcoliamo le predizioni (score da 0 a 1) per il Validation Set e per l'AmBe
print("Calcolo delle predizioni in corso...")
# Se hai appena trainato il modello, puoi usare direttamente linear_eval_model
# Altrimenti scommenta la riga sotto per ricaricarlo dal disco:
# linear_eval_model = keras.models.load_model("saved_models/CYGNO_Linear_Classifier_Final.keras")

X_ambe_test = ambe_images_real


scores_val = linear_eval_model.predict(X_val, batch_size=128)
scores_ambe = linear_eval_model.predict(X_ambe_test, batch_size=128)

# 2. Separiamo il validation set usando le label reali per vedere come si comporta il modello
# y_val == 0 sono gli Electronic Recoils (Background)
# y_val == 1 sono i Nuclear Recoils (Signal)
scores_val_bg = scores_val[y_val == 0]
scores_val_sig = scores_val[y_val == 1]

# 3. Creazione del Plot a Istogramma con Densità di Probabilità
plt.figure(figsize=(12, 7))

# density=True fa in modo che l'area totale dell'istogramma sia 1 (Densità di probabilità)
# alpha=0.5 rende i colori trasparenti per vedere le sovrapposizioni

# Plot Background (ER) - Ci aspettiamo che sia tutto schiacciato verso lo 0
plt.hist(scores_val_bg, bins=50, density=True, alpha=0.6, color='blue', 
         label='Validation: Background Puro (ER)')

# Plot Segnale (NR) - Ci aspettiamo che sia tutto schiacciato verso l'1
plt.hist(scores_val_sig, bins=50, density=True, alpha=0.6, color='red', 
         label='Validation: Segnale Puro (NR)')

# Plot AmBe (Miscela Reale) - Usiamo histtype='step' per fare solo il contorno e non pasticciare i colori
plt.hist(scores_ambe, bins=50, density=True, alpha=0.9, color='green', 
         histtype='step', linewidth=3, label='Test Reale: Sorgente AmBe (Miscela)')

# Dettagli estetici del grafico
plt.xlabel('Model Score (Probabilità di Nuclear Recoil)', fontsize=14)
plt.ylabel('Densità di Probabilità', fontsize=14)
plt.title('Distribuzione degli Score: Modello SimCLR su Dati CYGNO', fontsize=16, fontweight='bold')
plt.xlim(-0.05, 1.05) # Fissiamo l'asse X tra 0 e 1 (con un minimo di margine)
plt.legend(fontsize=12, loc='upper center')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 4. Statistiche veloci sull'AmBe
soglia_taglio = 0.8
nr_candidates = np.sum(scores_ambe > soglia_taglio)
print(f"\nStatistiche Dataset AmBe:")
print(f"- Tracce totali analizzate: {len(scores_ambe)}")
print(f"- Tracce classificate come probabili Nuclear Recoils (Score > {soglia_taglio}): {nr_candidates}")
print(f"- Frazione stimata di segnale (Alpha): {(nr_candidates / len(scores_ambe))*100:.2f}%")